# Calculate unit-wise difference between the Taunton-Worcester scenario and base

Aggregate differences by stroke unit.

## Code setup

In [1]:
import os
import numpy as np
import pandas as pd

In [2]:
dir_data = 'data'
dir_output = 'output'
# Scenario results:
file_results_comparison = 'results_full_comparison_taunworc_base.csv'
# Reference:
file_hospitals = 'stroke_hospitals_2025.csv'
# Output:

## Load data

### Scenario comparison results

In [3]:
p = os.path.join(dir_output, file_results_comparison)
df = pd.read_csv(p, index_col=0)

In [4]:
display(df.head(3))

,area_lower,Admissions_2025_Est,nearest_ivt_unit_base,transfer_unit_base,transfer_required_base,drip_ship_mt_travel_time_base,admissions_mt_base,transfers_mt_base,admissions_mt_transferloss0!6_base,transfers_mt_transferloss0!6_base,...,change_in_transfers_mt_transferloss0!6,change_in_admissions_weighted_drip_ship_mrs_0-2,change_in_admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2,change_in_admissions_weighted_drip_ship_mrs_0-2_transferloss0!6,change_in_admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2_transferloss0!6,change_in_drip_ship_mrs_0-2,change_in_lvo_drip_ship_ivt_mt_mrs_0-2,change_in_drip_ship_mrs_0-2_transferloss0!6,change_in_lvo_drip_ship_ivt_mt_mrs_0-2_transferloss0!6,changed
LSOA,,,,,,,,,,,,,,,,,,,,,
ADUR001A,Adur 001A,1.49403,BN25BE,BN25BE,False,10.9,0.149,0.000,0.149,0.000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,False
ADUR001B,Adur 001B,1.86762,BN25BE,BN25BE,False,11.9,0.187,0.000,0.187,0.000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,False
ADUR001C,Adur 001C,2.24110,BN112DH,BN25BE,True,95.0,0.224,0.224,0.134,0.134,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,False


In [5]:
print(df.columns)

Index(['area_lower', 'Admissions_2025_Est', 'nearest_ivt_unit_base',
       'transfer_unit_base', 'transfer_required_base',
       'drip_ship_mt_travel_time_base', 'admissions_mt_base',
       'transfers_mt_base', 'admissions_mt_transferloss0!6_base',
       'transfers_mt_transferloss0!6_base',
       'admissions_weighted_drip_ship_mrs_0-2_base',
       'admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2_base',
       'admissions_weighted_drip_ship_mrs_0-2_transferloss0!6_base',
       'admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2_transferloss0!6_base',
       'drip_ship_mrs_0-2_base', 'lvo_drip_ship_ivt_mt_mrs_0-2_base',
       'drip_ship_mrs_0-2_transferloss0!6_base',
       'lvo_drip_ship_ivt_mt_mrs_0-2_transferloss0!6_base',
       'nearest_ivt_unit_taunworc', 'transfer_unit_taunworc',
       'transfer_required_taunworc', 'drip_ship_mt_travel_time_taunworc',
       'admissions_mt_taunworc', 'transfers_mt_taunworc',
       'admissions_mt_transferloss0!6_taunworc',
       'transfer

Gather columns by type and without tailing scenario names (`_base`, `_taunworc`):

In [6]:
col_lsoa11nm = 'area_lower'
col_admissions = 'Admissions_2025_Est'
cols_unit_postcodes = ['nearest_ivt_unit', 'transfer_unit']
cols_mt_admissions = ['admissions_mt', 'admissions_mt_transferloss0!6']
cols_transfer = ['transfers_mt', 'transfers_mt_transferloss0!6']
cols_travel_time = ['drip_ship_mt_travel_time']
cols_outcome = [
    'drip_ship_mrs_0-2',
    'lvo_drip_ship_ivt_mt_mrs_0-2',
    'drip_ship_mrs_0-2_transferloss0!6',
    'lvo_drip_ship_ivt_mt_mrs_0-2_transferloss0!6',
]
cols_admissions_weighted_outcomes = [
    'admissions_weighted_drip_ship_mrs_0-2',
    'admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2',
    'admissions_weighted_drip_ship_mrs_0-2_transferloss0!6',
    'admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2_transferloss0!6',
]
cols_change = [
    'change_in_drip_ship_mt_travel_time',
    'change_in_drip_ship_mrs_0-2',
    'change_in_lvo_drip_ship_ivt_mt_mrs_0-2',
    'change_in_drip_ship_mrs_0-2_transferloss0!6',
    'change_in_lvo_drip_ship_ivt_mt_mrs_0-2_transferloss0!6',
    'changed'
]

cols_shared = [col_lsoa11nm, col_admissions] + cols_change

### Hospital reference

In [7]:
# Load hospital coordinates from CSV and create a GeoDataFrame
p = os.path.join(dir_data, file_hospitals)
df_hospitals = pd.read_csv(p, index_col='Postcode')

# Only keep hospitals in England:
df_hospitals = df_hospitals.loc[df_hospitals["Country"] == "England"]

In [8]:
display(df_hospitals.head(3))

,SSNAP name,Country,SSNAP Region,Easting,Northing,long,lat,Hospital_City,LSOA11CD,Use_IVT,Use_MT
Postcode,,,,,,,,,,,
OX39DU,John Radcliffe Hospital,England,"Buckinghamshire, Oxford and Berkshire",453940,207531,-1.219806,51.763871,Oxford,E01028534,1,1
RG15AN,Royal Berkshire Hospital,England,"Buckinghamshire, Oxford and Berkshire",472411,172962,-0.959330,51.451022,Reading,E01016414,1,0
HP112TT,Wycombe General Hospital,England,"Buckinghamshire, Oxford and Berkshire",486386,192694,-0.753428,51.626442,High Wycombe,E01017839,1,0


In [9]:
print(len(df_hospitals))

131


## Set up unit combinations

Make sure to separate pairs of units that are the same in one scenario but different in another. Use labels that start with '42' so they're not confused with a standard range index.

Use these combinations:

In [10]:
cols_units_here = [f'{c}_{s}' for c in cols_unit_postcodes
                   for s in ['base', 'taunworc']]
df_unit_combos = df[cols_units_here].drop_duplicates(ignore_index=True)
df_unit_combos['unit_combo'] = np.arange(len(df_unit_combos)) + 42000

Check labels for units that change with scenario:

In [11]:
m = ((df_unit_combos['nearest_ivt_unit_base'] !=
      df_unit_combos['nearest_ivt_unit_taunworc']) |
     (df_unit_combos['transfer_unit_base'] !=
      df_unit_combos['transfer_unit_taunworc']))

changed_unit_combos = df_unit_combos.loc[m, 'unit_combo'].values.tolist()

display(df_unit_combos.loc[m])

,nearest_ivt_unit_base,nearest_ivt_unit_taunworc,transfer_unit_base,transfer_unit_taunworc,unit_combo
57,WR51DD,WR51DD,B152TH,WR51DD,42057
60,GL537AN,GL537AN,BS105NB,WR51DD,42060
70,EX314JB,EX314JB,PL68DH,TA15DA,42070
83,TA15DA,TA15DA,BS105NB,TA15DA,42083
84,EX25DW,EX25DW,PL68DH,TA15DA,42084
86,DT12JY,DT12JY,SO166YD,TA15DA,42086
91,HR12ER,HR12ER,B152TH,WR51DD,42091
99,DT12JY,TA15DA,SO166YD,TA15DA,42099
100,DY12HQ,WR51DD,B152TH,WR51DD,42100


For example, the combination of nearest-transfer unit 'TA15DA'-'TA15DA' appears twice in the `taunworc` columns, but these patients must be kept separate for the comparison with the base scenario because of the difference in their `base` columns.

Merge these labels into the main dataframe:

In [12]:
df = pd.merge(df, df_unit_combos, on=cols_units_here, how='left')

Check that labels are as expected:

In [127]:
cols_to_display = ['nearest_ivt_unit_base', 'nearest_ivt_unit_taunworc',
                   'transfer_unit_base', 'transfer_unit_taunworc',
                   'unit_combo', col_admissions]

display(df[df['unit_combo'].isin(changed_unit_combos)][cols_to_display]    
        .sample(5, random_state=42))

,nearest_ivt_unit_base,nearest_ivt_unit_taunworc,transfer_unit_base,transfer_unit_taunworc,unit_combo,Admissions_2025_Est
18817,EX314JB,EX314JB,PL68DH,TA15DA,42070,3.36165
18882,DT12JY,DT12JY,SO166YD,TA15DA,42086,0.37348
28290,GL537AN,GL537AN,BS105NB,WR51DD,42060,1.12055
17597,TA15DA,TA15DA,BS105NB,TA15DA,42083,4.10862
27880,TA15DA,TA15DA,BS105NB,TA15DA,42083,4.10862


## Define functions

Create some functions for the calculations.

In [21]:
def calculate_admissions_by_unit(
        df,
        cols_admissions
    ):
    """
    """
    df_admissions = (
        df[['unit_combo'] + cols_admissions].groupby('unit_combo').sum())
    
    df_admissions = df_admissions.fillna(0.0)
    df_admissions = df_admissions.round(1)
    return df_admissions

In [41]:
def calculate_outcomes_by_unit(
    df,  # by LSOA
    cols_admissions_weighted_outcomes,
    col_admissions
    ):
    """
    """
    # First sum by all patients:
    df_unit_outcomes = (
        df[['unit_combo', col_admissions] + cols_admissions_weighted_outcomes]
        .groupby('unit_combo').sum()
    )
    # Divide outcomes by the direct admissions to each unit:
    df_unit_outcomes[cols_admissions_weighted_outcomes] = (
        df_unit_outcomes[cols_admissions_weighted_outcomes]
            .div(df_unit_outcomes[col_admissions], axis='rows')
    )
    df_unit_outcomes = df_unit_outcomes.drop(col_admissions, axis='columns')
    df_unit_outcomes = df_unit_outcomes.round(5)
    return df_unit_outcomes


Gather these separate functions into a main one:

In [42]:
def main(
        df,
        scenario,
        df_unit_combos,
        mt_rate,
        transfer_rates,
        cols_shared,
        col_admissions,
        cols_unit_postcodes,
        cols_outcome,
    ):
    """
    """
    print(scenario)
    cols = (['unit_combo'] + cols_shared +
            [c for c in df.columns if c.endswith(scenario)])
    df_here = df[cols]
    rename_dict = dict([(c, c.replace(f'_{scenario}', ''))
                        for c in df_here.columns])
    df_here = df_here.rename(columns=rename_dict)

    # Link each first unit to its MT unit:
    df_unit_links = df_unit_combos.copy()
    rename_dict = dict([(c, c.replace(f'_{scenario}', ''))
                        for c in df_unit_links.columns])
    df_unit_links = df_unit_links.rename(columns=rename_dict)
    df_unit_links = (
        df_unit_links.set_index('unit_combo')
        [['nearest_ivt_unit', 'transfer_unit']]
    )

    # Get unit names and cities:
    df_unit_info = df_unit_links.copy().reset_index()
    cols_unit_info = ['SSNAP name', 'Hospital_City']
    df_unit_info = pd.merge(
        df_unit_info,
        df_hospitals[cols_unit_info].reset_index(),
        left_on='nearest_ivt_unit',
        right_on='Postcode',
        how='left'
    ).drop('Postcode', axis='columns')
    df_unit_info = pd.merge(
        df_unit_info,
        df_hospitals[cols_unit_info].reset_index(),
        left_on='transfer_unit',
        right_on='Postcode',
        how='left',
        suffixes=(None, '_transfer_unit')
    ).drop('Postcode', axis='columns')
    df_unit_info = df_unit_info.set_index('unit_combo')
    print('df_unit_info:', df_unit_info.columns)
    # display(df_unit_info)
    
    # Sum admissions to each unit by scenario
    df_admissions = calculate_admissions_by_unit(
        df_here,
        [col_admissions] + cols_mt_admissions + cols_transfer
    )
    print('df_admissions:', df_admissions.columns)
    # display(df_admissions)

    # Group by unit:
    df_unit_outcomes = calculate_outcomes_by_unit(
        df_here,
        cols_admissions_weighted_outcomes,
        col_admissions
    )
    print('df_unit_outcomes:', df_unit_outcomes.columns)
    # display(df_unit_outcomes)

    # Gather all results:
    comparison_by_unit_combo = pd.concat(
        (df_unit_info, df_admissions, df_unit_outcomes),
        axis='columns'
    )
    return comparison_by_unit_combo

## Set up scenarios

Run calculations separately for each scenario, otherwise it's difficult to see what's happening.

This cell prints the columns that are created by each function in main().

In [43]:
mt_rate = 0.1
transfer_rates = [1.0, 0.6]  # should match existing results files.

comparison_by_unit_combo_base = main(
    df,
    'base',
    df_unit_combos,
    mt_rate,
    transfer_rates,
    cols_shared,
    col_admissions,
    cols_unit_postcodes,
    cols_outcome,
)
print('')
comparison_by_unit_combo_taunworc = main(
    df,
    'taunworc',
    df_unit_combos,
    mt_rate,
    transfer_rates,
    cols_shared,
    col_admissions,
    cols_unit_postcodes,
    cols_outcome,
)

base
df_unit_info: Index(['nearest_ivt_unit', 'transfer_unit', 'SSNAP name', 'Hospital_City',
       'SSNAP name_transfer_unit', 'Hospital_City_transfer_unit'],
      dtype='str')
df_admissions: Index(['Admissions_2025_Est', 'admissions_mt', 'admissions_mt_transferloss0!6',
       'transfers_mt', 'transfers_mt_transferloss0!6'],
      dtype='str')
df_unit_outcomes: Index(['admissions_weighted_drip_ship_mrs_0-2',
       'admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2',
       'admissions_weighted_drip_ship_mrs_0-2_transferloss0!6',
       'admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2_transferloss0!6'],
      dtype='str')
taunworc
df_unit_info: Index(['nearest_ivt_unit', 'transfer_unit', 'SSNAP name', 'Hospital_City',
       'SSNAP name_transfer_unit', 'Hospital_City_transfer_unit'],
      dtype='str')
df_admissions: Index(['Admissions_2025_Est', 'admissions_mt', 'admissions_mt_transferloss0!6',
       'transfers_mt', 'transfers_mt_transferloss0!6'],
      dtype='str')
df_unit_outc

## Check for changes between scenarios

Pick out the units in the base scenario for LSOA where the scenario affects the choice of unit.

In [110]:
m = (comparison_by_unit_combo_base.fillna(0.0) ==
     comparison_by_unit_combo_taunworc.fillna(0.0)).all(axis='columns')

units_with_change = comparison_by_unit_combo_base.loc[~m, 'nearest_ivt_unit'].tolist()
mt_units_with_change = comparison_by_unit_combo_base.loc[~m, 'transfer_unit'].tolist()

In [111]:
display(df_hospitals.loc[
    sorted(list(set(units_with_change) | set(mt_units_with_change))),
    ['SSNAP name', 'Hospital_City']
    ])

,SSNAP name,Hospital_City
Postcode,,
B152TH,Queen Elizabeth Hospital Edgbaston,Birmingham
BS105NB,Bristol and Weston SU,North Bristol
DT12JY,Dorset County Hospital,Dorchester
DY12HQ,Russells Hall Hospital,Dudley
EX25DW,Royal Devon and Exeter Hospital,Exeter
EX314JB,North Devon District Hospital,Barnstaple
GL537AN,Cheltenham & Gloucester Hospitals,Cheltenham
HR12ER,Hereford County Hospital,Hereford
PL68DH,Derriford Hospital,Plymouth


See that the expected units have changed - ones near Taunton and Worcester.

## Gather and simplify info

Place all results in same dataframe:

In [47]:
rename_base = dict([(c, f'{c}_base')
                    for c in comparison_by_unit_combo_base.columns])
rename_taunworc = dict([(c, f'{c}_taunworc')
                       for c in comparison_by_unit_combo_taunworc.columns])

comparison_by_unit_combo = pd.concat(
    (comparison_by_unit_combo_base.rename(columns=rename_base),
     comparison_by_unit_combo_taunworc.rename(columns=rename_taunworc)),
    axis='columns'
).sort_index()

Reorder columns:

In [48]:
col_order = [
    'nearest_ivt_unit', 'transfer_unit', 'SSNAP name', 'Hospital_City',
    'SSNAP name_transfer_unit', 'Hospital_City_transfer_unit',
]
col_order = [f'{c}_{s}' for c in col_order for s in ['base', 'taunworc']]
col_order += [c for c in comparison_by_unit_combo.columns if c not in col_order]

comparison_by_unit_combo = comparison_by_unit_combo[col_order]

Check that there is one entry for each combination of units:

In [49]:
print(len(comparison_by_unit_combo) == len(df_unit_combos))

True


Some extra calculations:

In [124]:
def calculate_extra_bits(df):
    # Changes between scenarios
    # First unit admissions doesn't depend on transfer rate:
    c = 'Admissions_2025_Est'
    df['change_in_first_admissions'] = (
        df[f'{c}_taunworc']
        - df[f'{c}_base']
    ).round(1)
    
    for t in transfer_rates:
        s = '' if t == 1.0 else f'_transferloss{t}'.replace('.', '!')
        c = 'admissions_mt'
        df[f'change_in_all_MT_admissions{s}'] = (
            df[f'{c}{s}_taunworc']
            - df[f'{c}{s}_base']
        ).round(1)
        c = 'admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2'
        df[f'change_in_outcome_lvo_mrs_0-2{s}'] = (
            df[f'{c}{s}_taunworc']
            - df[f'{c}{s}_base']
        ).round(5)

    if 'transfer_unit_base' in df.columns:
        # Find MT units with a change in total admissions
        mt_units_with_change = sorted(list(set(df.loc[
            df['change_in_outcome_lvo_mrs_0-2'] != 0,
            ['transfer_unit_base', 'transfer_unit_taunworc']
            ].values.flatten())))
        
        # Indicate whether each MT unit's admissions changed from any IVT unit
        df['mt_unit_admissions_changed_by_any_ivt_unit'] = (
            df['transfer_unit_base'].isin(mt_units_with_change) |
            df['transfer_unit_taunworc'].isin(mt_units_with_change) 
        )
    return df

In [ ]:
comparison_by_unit_combo = calculate_extra_bits(comparison_by_unit_combo)

In [56]:
col_tups = (
    ('change_in_first_admissions', 'Admissions_2025_Est_taunworc', 'Admissions_2025_Est_base'),
    ('change_in_all_MT_admissions', 'admissions_mt_taunworc', 'admissions_mt_base'),
    ('change_in_all_MT_admissions_transferloss0!6', 'admissions_mt_transferloss0!6_taunworc', 'admissions_mt_transferloss0!6_base'),
    ('change_in_outcome_lvo_mrs_0-2', 'admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2_taunworc', 'admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2_base'),
    ('change_in_outcome_lvo_mrs_0-2_transferloss0!6', 'admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2_transferloss0!6_taunworc', 'admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2_transferloss0!6_base'),
)

for cols in col_tups:
    m = comparison_by_unit_combo[cols[0]] != 0.0
    display(comparison_by_unit_combo.loc[m, list(cols)])

,change_in_first_admissions,Admissions_2025_Est_taunworc,Admissions_2025_Est_base
unit_combo,,,


,change_in_all_MT_admissions,admissions_mt_taunworc,admissions_mt_base
unit_combo,,,


,change_in_all_MT_admissions_transferloss0!6,admissions_mt_transferloss0!6_taunworc,admissions_mt_transferloss0!6_base
unit_combo,,,
42057,28.0,70.0,42.0
42083,37.4,93.6,56.2
42099,0.2,0.6,0.4
42100,0.2,0.5,0.3


,change_in_outcome_lvo_mrs_0-2,admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2_taunworc,admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2_base
unit_combo,,,
42057,0.05567,0.42711,0.37144
42060,0.00922,0.37318,0.36396
42070,0.01428,0.35243,0.33815
42083,0.07129,0.42331,0.35202
42084,0.01035,0.36766,0.35731
42086,0.00057,0.35109,0.35052
42091,0.01516,0.36442,0.34926
42099,0.07244,0.41561,0.34317
42100,0.04901,0.42095,0.37194


,change_in_outcome_lvo_mrs_0-2_transferloss0!6,admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2_transferloss0!6_taunworc,admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2_transferloss0!6_base
unit_combo,,,
42057,0.07471,0.42711,0.35240
42060,0.00553,0.35351,0.34798
42070,0.00858,0.34049,0.33191
42083,0.08322,0.42331,0.34009
42084,0.00622,0.34998,0.34376
42086,0.00035,0.33958,0.33923
42091,0.00910,0.34791,0.33881
42099,0.08205,0.41561,0.33356
42100,0.06940,0.42095,0.35155


Check which MT units change between scenarios:

In [84]:
mt_units_with_change

['B152TH', 'BS105NB', 'PL68DH', 'SO166YD', 'TA15DA', 'WR51DD']

## Save results

Save one copy with all unit combos.

In [91]:
p = os.path.join(dir_output, 'comparison_by_unit_combo_base_taunworc.csv')
comparison_by_unit_combo.to_csv(p, index=True)

## Combine units

Admissions can just be summed.

Admission-weighted outcomes need to be weighted more carefully.

In [120]:
def combine_units_admissions_weighted_outcomes(
        df,  # by unit, units may appear more than once
        cols_admissions_weighted_outcomes,
        col_admissions,
        col_unit
    ):
    df = df.copy()
    df[cols_admissions_weighted_outcomes] = (
        df[cols_admissions_weighted_outcomes]
            .multiply(df[col_admissions], axis='rows')
    )
    cols = [col_unit, col_admissions] + cols_admissions_weighted_outcomes
    df_here = df[cols].groupby(col_unit).sum()
    # Divide outcomes by the direct admissions to each unit:
    df_here[cols_admissions_weighted_outcomes] = (
        df_here[cols_admissions_weighted_outcomes]
            .div(df_here[col_admissions], axis='rows')
    )
    df_here = df_here.drop(col_admissions, axis='columns')
    df_here = df_here.round(5)

    return df_here

In [115]:
cols_unit = ['nearest_ivt_unit', 'transfer_unit']
cols_to_sum = []

dict_by_unit = {}
for s in ['base', 'taunworc']:
    # Set up columns:
    cols_unit_here = [f'{c}_{s}' for c in cols_unit]
    cols_scen = [c for c in comparison_by_unit_combo.columns if s in c]
    cols_adweight = [c for c in cols_scen if 'weighted' in c]
    cols_ad = [c for c in cols_scen if
                   ((('dmissions' in c) | ('transfers_mt' in c)) &
                    (c not in cols_adweight))]
    col_ad = f'{col_admissions}_{s}'
    # Remaining columns should be nearest_ivt_unit and transfer_unit
    # and their SSNAP names and hospital cities.
    cols_left = list(set(cols_scen) - (set(cols_adweight) | set(cols_to_sum)))

    for col_unit in cols_unit_here:
        # Stroke unit info:
        if 'transfer' in col_unit:
            cols_this_unit = [f'SSNAP name_transfer_unit_{s}',
                              f'Hospital_City_transfer_unit_{s}']
        else:
            cols_this_unit = [f'SSNAP name_{s}', f'Hospital_City_{s}']
        df_units = comparison_by_unit_combo[[col_unit] + cols_this_unit].drop_duplicates(ignore_index=True)
        df_units = df_units.set_index(col_unit)

        df_ad = comparison_by_unit_combo[[col_unit] + cols_ad].copy()
        df_ad = df_ad.groupby(col_unit).sum()

        df_adweight = comparison_by_unit_combo[[col_unit, col_ad] + cols_adweight].copy()
        df_adweight = combine_units_admissions_weighted_outcomes(
            df_adweight,  # by unit, units may appear more than once
            cols_adweight,
            col_ad,
            col_unit
        )

        df_here = pd.concat((df_units, df_ad, df_adweight), axis='columns')

        dict_by_unit[col_unit] = df_here.copy()

Combine scenarios.

Keep the nearest IVT unit and transfer unit data separate because they're not always the same pair between scenarios.

In [117]:
comparison_by_nearest_ivt_unit = pd.concat(
    (dict_by_unit['nearest_ivt_unit_base'],
     dict_by_unit['nearest_ivt_unit_taunworc']),
    axis='columns'
)
comparison_by_transfer_unit = pd.concat(
    (dict_by_unit['transfer_unit_base'],
     dict_by_unit['transfer_unit_taunworc']),
    axis='columns'
)

In [125]:
comparison_by_nearest_ivt_unit = calculate_extra_bits(comparison_by_nearest_ivt_unit)
comparison_by_transfer_unit = calculate_extra_bits(comparison_by_transfer_unit)

In [126]:
col_tups = (
    ('change_in_first_admissions', 'Admissions_2025_Est_taunworc', 'Admissions_2025_Est_base'),
    ('change_in_all_MT_admissions', 'admissions_mt_taunworc', 'admissions_mt_base'),
    ('change_in_all_MT_admissions_transferloss0!6', 'admissions_mt_transferloss0!6_taunworc', 'admissions_mt_transferloss0!6_base'),
    ('change_in_outcome_lvo_mrs_0-2', 'admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2_taunworc', 'admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2_base'),
    ('change_in_outcome_lvo_mrs_0-2_transferloss0!6', 'admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2_transferloss0!6_taunworc', 'admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2_transferloss0!6_base'),
)

for cols in col_tups:
    m = comparison_by_nearest_ivt_unit[cols[0]] != 0.0
    display(comparison_by_nearest_ivt_unit.loc[m, list(cols)])

    
for cols in col_tups:
    m = comparison_by_transfer_unit[cols[0]] != 0.0
    display(comparison_by_transfer_unit.loc[m, list(cols)])

,change_in_first_admissions,Admissions_2025_Est_taunworc,Admissions_2025_Est_base
DY12HQ,-4.9,1035.5,1040.4
WR51DD,4.9,705.5,700.6
TA15DA,6.3,942.4,936.1
DT12JY,-6.3,697.4,703.7


,change_in_all_MT_admissions,admissions_mt_taunworc,admissions_mt_base
DY12HQ,-0.5,103.5,104.0
WR51DD,0.5,70.5,70.0
TA15DA,0.6,94.2,93.6
DT12JY,-0.6,69.7,70.3


,change_in_all_MT_admissions_transferloss0!6,admissions_mt_transferloss0!6_taunworc,admissions_mt_transferloss0!6_base
DY12HQ,-0.3,62.1,62.4
WR51DD,28.5,70.5,42.0
TA15DA,38.0,94.2,56.2
DT12JY,-0.4,41.8,42.2


,change_in_outcome_lvo_mrs_0-2,admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2_taunworc,admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2_base
DY12HQ,0.00004,0.381060,0.381017
WR51DD,0.05563,0.427067,0.371440
GL537AN,0.00922,0.373180,0.363960
EX314JB,0.01428,0.352430,0.338150
TA15DA,0.07124,0.423259,0.352020
EX25DW,0.01035,0.367660,0.357310
DT12JY,0.00064,0.351090,0.350454
HR12ER,0.01516,0.364420,0.349260


,change_in_outcome_lvo_mrs_0-2_transferloss0!6,admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2_transferloss0!6_taunworc,admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2_transferloss0!6_base
DY12HQ,0.00003,0.358800,0.358766
WR51DD,0.07467,0.427067,0.352400
GL537AN,0.00553,0.353510,0.347980
EX314JB,0.00858,0.340490,0.331910
TA15DA,0.08317,0.423259,0.340090
EX25DW,0.00622,0.349980,0.343760
DT12JY,0.00040,0.339580,0.339179
HR12ER,0.00910,0.347910,0.338810


,change_in_first_admissions,Admissions_2025_Est_taunworc,Admissions_2025_Est_base
SO166YD,-703.7,4884.8,5588.5
BS105NB,-1734.2,2197.4,3931.6
B152TH,-1222.1,5871.8,7093.9
PL68DH,-1095.9,2317.7,3413.6
WR51DD,NaN,2020.2,NaN
TA15DA,NaN,2735.7,NaN


,change_in_all_MT_admissions,admissions_mt_taunworc,admissions_mt_base
SO166YD,-70.3,488.5,558.8
BS105NB,-173.4,219.7,393.1
B152TH,-122.2,587.1,709.3
PL68DH,-109.6,231.8,341.4
WR51DD,NaN,202.0,NaN
TA15DA,NaN,273.5,NaN


,change_in_all_MT_admissions_transferloss0!6,admissions_mt_transferloss0!6_taunworc,admissions_mt_transferloss0!6_base
SO166YD,-42.2,326.2,368.4
BS105NB,-104.1,185.6,289.7
B152TH,-73.3,386.7,460.0
PL68DH,-65.7,172.1,237.8
WR51DD,NaN,149.4,NaN
TA15DA,NaN,201.7,NaN


,change_in_outcome_lvo_mrs_0-2,admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2_taunworc,admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2_base
SO166YD,0.00367,0.379570,0.375904
BS105NB,0.02049,0.403973,0.383481
B152TH,0.00411,0.385920,0.381811
PL68DH,0.00949,0.378692,0.369205
WR51DD,NaN,0.389759,NaN
TA15DA,NaN,0.379987,NaN


,change_in_outcome_lvo_mrs_0-2_transferloss0!6,admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2_transferloss0!6_taunworc,admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2_transferloss0!6_base
SO166YD,0.00321,0.364671,0.361461
BS105NB,0.02333,0.396622,0.373288
B152TH,0.00358,0.367449,0.363866
PL68DH,0.01039,0.371083,0.360689
WR51DD,NaN,0.377766,NaN
TA15DA,NaN,0.370951,NaN
